<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 10 · REAL-TIME ANALYTICS WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Control Storage Cost and the Online Data Lifecycle</div>
  <p style="color:#475569;font-size:15px;line-height:1.7;max-width:920px;margin:0">Compare table-level compression with controlled input, maintain a rolling window of complete date Partitions, observe an out-of-window write, and recover one manually dropped Partition.</p>
  <span style="display:inline-block;border:1px solid #99f6e4;border-radius:4px;background:#f0fdfa;color:#115e59;padding:6px 10px;margin-top:14px;font-size:12px">Compression · Range Partition · Dynamic Partitioning · Retention · Recycle bin · RECOVER</span>
</div>

A commerce event table keeps growing. The team needs to reduce the bytes stored per event, keep only a bounded online time window, and recover from an accidental Partition drop. Work through four steps:

1. Compare compression choices on one structured event-log input.
2. Create a rolling Dynamic Partition window around the current hour.
3. Return after the next hourly boundary and observe an expired Partition leave the online table.
4. Drop and recover a separate manual Partition.

**Run Sections 1–2 in order, then pause at Section 3 until the next hourly boundary.** Do not use Run All across that pause. Each setup section resets only the objects it owns: each Section 1 load cell resets its own compression table, Section 2 resets `m10_retained_events`, and Section 4 resets `m10_recovery_events`. The Lab reads `events_modelled` but never changes it. The Lab uses the single-node `doris` environment from Levels 1–8 and does not access AWS S3, HDFS, or remote storage.

### Initialize the Lab

Run the next cell before Section 1 and again after restarting the Jupyter kernel. It locates the course root and loads the shared `DorisLab` helper. It does not start Docker or change Doris data.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab
from doris_course.ui import card

lab = DorisLab(lab_dir=COURSE_ROOT);

## 1. Compare compression choices on structured event logs

**Need:** a commerce analytics team stores many event-log messages. Disk space matters, but frequent reads also consume CPU to decompress Pages. Which table-level compression setting deserves further testing?

Start or reuse the existing local Doris 4.1.3 container and the `doris_course` Database. Read one day from `events_modelled` without changing it. Each copied row receives a deterministic synthetic `event_context`: its `event_id` makes the whole message unique, while JSON field names and descriptive text repeat. This is a controlled teaching input, not an existing source field or a measured production workload.

The input interval is **[2020-03-01 00:00:00, 2020-03-02 00:00:00)**. The first boundary is included and the second is excluded. Check only the known row count before rebuilding this Lab's tables.


In [ ]:
lab.start_container("doris")
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

source_ready = lab.sql("""
SELECT COUNT(*) AS event_count
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Compression input: one complete day")
assert int(source_ready.iloc[0]["event_count"]) == 75_259,     "Expected 75,259 events. Complete Lab 4 before continuing."

**Expected result:** `event_count` is **75,259**. The check reads the source but does not modify it.

### Build three copies of the same input

Each of the next three cells drops, creates, and loads only its own `m10_*` table. Rerun a complete cell if it fails; the sequence is repeatable but not one atomic transaction. The tables have identical columns, Duplicate Key order, Range Partition, four Buckets, and one replica per Tablet. Only the `compression` table property differs: `NO_COMPRESSION` (encoded data without Page compression), `LZ4F`, or the build's default `ZSTD`. `NO_COMPRESSION` is the spelling accepted by the tested Doris 4.1.3 build.

Each `INSERT ... SELECT` copies the same event rows and generates the **same** structured message. The whole message varies with `event_id`, but its keys and much of its text do not. Keeping the input and layout equal lets the reported table sizes answer a specific question about the compression choice.


In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_events_none FORCE")
lab.execute("""
CREATE TABLE m10_events_none (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    event_context STRING NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
PARTITION BY RANGE(event_time) (
    PARTITION p20200301 VALUES [('2020-03-01'), ('2020-03-02'))
)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES (
    "replication_num" = "1",
    "compression" = "NO_COMPRESSION"
)
""")
lab.insert("""
INSERT INTO m10_events_none
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue,
       CONCAT(
           '{"source":"commerce-web","schema":"event-v1",',
           '"pipeline":"real-time-analytics",',
           '"message":"A customer event entered the commerce analytics pipeline.",',
           '"event_id":"', CAST(event_id AS STRING), '",',
           '"event_type":"', event_type, '",',
           '"region":"', region, '",',
           '"product_id":"', CAST(product_id AS STRING), '"}'
       ) AS event_context
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Load the NO_COMPRESSION table", expected_rows=75_259);

In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_events_lz4f FORCE")
lab.execute("""
CREATE TABLE m10_events_lz4f (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    event_context STRING NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
PARTITION BY RANGE(event_time) (
    PARTITION p20200301 VALUES [('2020-03-01'), ('2020-03-02'))
)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES (
    "replication_num" = "1",
    "compression" = "LZ4F"
)
""")
lab.insert("""
INSERT INTO m10_events_lz4f
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue,
       CONCAT(
           '{"source":"commerce-web","schema":"event-v1",',
           '"pipeline":"real-time-analytics",',
           '"message":"A customer event entered the commerce analytics pipeline.",',
           '"event_id":"', CAST(event_id AS STRING), '",',
           '"event_type":"', event_type, '",',
           '"region":"', region, '",',
           '"product_id":"', CAST(product_id AS STRING), '"}'
       ) AS event_context
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Load the LZ4F table", expected_rows=75_259);

In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_events_zstd FORCE")
lab.execute("""
CREATE TABLE m10_events_zstd (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    event_context STRING NOT NULL
)
DUPLICATE KEY(event_time, event_id, user_id)
PARTITION BY RANGE(event_time) (
    PARTITION p20200301 VALUES [('2020-03-01'), ('2020-03-02'))
)
DISTRIBUTED BY HASH(user_id) BUCKETS 4
PROPERTIES (
    "replication_num" = "1",
    "compression" = "ZSTD"
)
""")
lab.insert("""
INSERT INTO m10_events_zstd
SELECT event_time, event_id, user_id, event_type, region, product_id, revenue,
       CONCAT(
           '{"source":"commerce-web","schema":"event-v1",',
           '"pipeline":"real-time-analytics",',
           '"message":"A customer event entered the commerce analytics pipeline.",',
           '"event_id":"', CAST(event_id AS STRING), '",',
           '"event_type":"', event_type, '",',
           '"region":"', region, '",',
           '"product_id":"', CAST(product_id AS STRING), '"}'
       ) AS event_context
FROM events_modelled
WHERE event_time >= '2020-03-01 00:00:00'
  AND event_time <  '2020-03-02 00:00:00'
""", title="Load the ZSTD table", expected_rows=75_259);

### Inspect the values, then compare storage reports

A column with only a few distinct values, such as `event_type`, may be compact after **column encoding**. That does not guarantee a large table-size reduction from Page compression. Conversely, each full `event_context` is unique, yet the repeated JSON keys and message text provide common **byte patterns**. Inspect both shapes before reading the table-size comparison. The three choices have different operating goals:

| Choice | What Doris does at the Page layer | Main trade-off | Typical reason to choose it |
|---|---|---|---|
| `NO_COMPRESSION` | Keeps column encoding but skips Page compression | Avoids Page compression/decompression work, but usually reads and stores more bytes | The payload is already compressed, or CPU cost matters more than storage |
| `LZ4F` | Compresses each encoded Page with the LZ4 Frame format | Very fast compression and decompression with a moderate reduction in bytes | Frequently queried data where low decompression overhead matters |
| `ZSTD` | Compresses each encoded Page with Zstandard | Usually reduces more bytes and performs more compression work; decompression remains fast | A balance between storage efficiency and analytical read performance; it is the default in this build |

These are selection characteristics, not conclusions measured by this small Lab. The experiment directly observes value counts, equal committed row counts, and Doris's current stored-size report. It does not measure CPU consumption or production query latency.

Committed rows become queryable before all Tablet size and row-count reports reach the Frontend (FE). The helper first checks each table with `SELECT COUNT(*)`, then waits for `SHOW DATA` to report 75,259 rows and a nonzero size. The wait is bounded at 180 seconds and fails with the latest metadata instead of running forever.

The final table combines four pieces of evidence:

- `SHOW CREATE TABLE` for a non-default compression property;
- FE `default_compression_type` when Doris normalizes the default property out of the returned DDL;
- `SHOW DATA`, `SHOW PARTITIONS`, and `SHOW TABLETS` for the current physical report;
- the reported percentage saved relative to `m10_events_none`, calculated from the same metadata snapshot.


### See two event types and a shared message prefix

The source day contains only two `event_type` values. The query below shows each value and its row count. It also counts **distinct complete** `event_context` strings and displays a short prefix from one example per type. Look for the same `source`, `schema`, and `pipeline` keys in both prefixes. `COUNT(DISTINCT ...)` counts values; it does not remove rows, reveal physical column sizes, or prove which encoding Doris chose.


In [ ]:
value_patterns = lab.sql("""
SELECT event_type, COUNT(*) AS event_count,
       COUNT(DISTINCT event_context) AS distinct_contexts,
       SUBSTRING(MIN(event_context), 1, 92) AS shared_prefix
FROM m10_events_none
GROUP BY event_type
ORDER BY event_type
""", title="Two event types, unique messages, shared bytes")
assert value_patterns["event_type"].tolist() == ["cart", "purchase"]
assert value_patterns["event_count"].astype(int).tolist() == [56_961, 18_298]
assert (value_patterns["event_count"] == value_patterns["distinct_contexts"]).all()
assert value_patterns["shared_prefix"].nunique() == 1


**Expected result:** the two rows are `cart` (**56,961**) and `purchase` (**18,298**). For each type, `distinct_contexts = event_count`: no complete generated message repeats. The displayed prefixes share the JSON keys and descriptive structure even though the full strings differ. The two event types can be cheap to represent through column encoding, while repeated substrings in the unique messages offer a separate opportunity for Page compression. Neither result reports a per-column physical compression ratio.

Next compare the three tables containing this same input. `SHOW DATA` reports each **whole table**, so the result cannot isolate how many bytes `event_type` or `event_context` saved individually.


In [ ]:
compression_tables = [
    "m10_events_none",
    "m10_events_lz4f",
    "m10_events_zstd",
]
lab.wait_for_storage_reports(
    compression_tables,
    expected_rows=75_259,
    timeout_seconds=180,
)
storage_comparison = lab.compression_storage_summary(
    compression_tables,
    baseline_table="m10_events_none",
);

**How to interpret the result:**

- `effective compression` confirms that the comparison actually uses `NO_COMPRESSION`, `LZ4F`, and `ZSTD`.
- All three rows show 75,259 logical rows, one Partition, four Tablets, and one replica per Tablet. Together with the identical `INSERT ... SELECT` input, these equal values establish the controlled comparison.
- `reported stored size` is the amount Doris currently reports for this sample. `reported space saved vs baseline` turns the same values into a comparison against `NO_COMPRESSION`; a positive percentage means fewer reported bytes.
- `ZSTD` may say `resolved from = FE default`. In this build, `SHOW CREATE TABLE` omits a property equal to the current default even though the visible `CREATE TABLE` statement explicitly requested `ZSTD`.

Compare the result with the two-row value-pattern check: the generated context has no duplicate complete strings, but its repeated JSON keys and message text provide repeated bytes for Page compression to exploit. `event_type` may already be compact after column encoding; only an actual storage measurement can establish the complete-table difference. File headers, indexes, Segment metadata, four Tablets, and rounding still affect the table-level totals. The measured difference belongs to this combined table and layout; it does not isolate `event_context` or prove a per-column compression ratio. On the verified local 4.1.3 build, the generated shared-text input produced a clear table-size reduction for both `LZ4F` and `ZSTD`: the reported sizes were about 18.0 MB for `NO_COMPRESSION`, 2.4 MB for `LZ4F` (86.6% less), and 1.9 MB for `ZSTD` (89.7% less). Your exact values may differ.

**Make a choice from the evidence:** `NO_COMPRESSION` is an encoded-table baseline, not the raw uncompressed size of the source rows. The percentage shown is only the difference between Doris's reported table sizes. Compare both `LZ4F` and `ZSTD` with the encoded `NO_COMPRESSION` baseline. A large reduction would demonstrate the effect of this shared-text workload, not a universal ratio for commerce events. The smaller of the two is a storage result for this sample; it does not prove that it wins on CPU cost or query latency. You would need a representative production-sized load and repeated workload measurements before making that claim. A small difference is a useful result: **do not tune a compression property just because another algorithm exists.**


## 2. Create an hourly Dynamic Partition window

Compression reduces stored bytes; retention decides **which complete time ranges stay online**. This experiment waits for the clock to advance rather than editing a retention rule or issuing `DROP PARTITION`.

The table uses an `event_time` **DATETIME** Partition key and `Asia/Shanghai` for both the Doris session clock and the scheduler. `time_unit = HOUR`, `start = -3`, and `end = 1` create the previous three complete hours, the current hour, and one future hour. `create_history_partition = true` makes the older hours available immediately. The window contains **five hourly Partitions**, not a rolling three-hour timer on each row.

The next cell rebuilds only `m10_retained_events` and waits for its five initial Partitions. Run it once to start a new observation. Do not rerun it when you return after the next hour: that would delete and recreate the evidence.


In [ ]:
lab.execute("SET time_zone = 'Asia/Shanghai'")
lab.execute("DROP TABLE IF EXISTS m10_retained_events FORCE")
lab.execute("""
CREATE TABLE m10_retained_events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    note VARCHAR(64) NOT NULL
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) ()
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "1",
    "dynamic_partition.enable" = "true",
    "dynamic_partition.time_unit" = "HOUR",
    "dynamic_partition.start" = "-3",
    "dynamic_partition.end" = "1",
    "dynamic_partition.prefix" = "p",
    "dynamic_partition.buckets" = "1",
    "dynamic_partition.create_history_partition" = "true",
    "dynamic_partition.time_zone" = "Asia/Shanghai"
)
""")
lab.wait_for_partition_count(
    "m10_retained_events", expected_count=5, timeout_seconds=60
);

Insert an event from three hours ago into the **oldest retained hourly Partition**. Add two newer events so you can see what remains after the next hour boundary. The scheduler, not an event-level timer, decides when the oldest whole Partition is removed.

The marker query prints its real `event_time` and `eligible_at`: the next whole-hour boundary when its Partition falls outside `start = -3`. Save that displayed time. `SHOW PARTITIONS` below shows the actual half-open hourly ranges and their rows. If you ran setup at 16:xx, the oldest Partition begins at 13:00, and its earliest eligible removal is 17:00 in Asia/Shanghai. The default FE scheduler may act later.


In [ ]:
lab.insert("""
INSERT INTO m10_retained_events VALUES
    (DATE_SUB(NOW(), INTERVAL 3 HOUR), 1001, 'oldest retained hour'),
    (DATE_SUB(NOW(), INTERVAL 2 HOUR), 1002, 'newer hour'),
    (NOW(),                             1003, 'current hour')
""", title="Insert three known hourly events", expected_rows=3)

lab.sql("""
SELECT event_time, event_id, note,
       DATE_ADD(DATE_TRUNC(event_time, 'hour'), INTERVAL 4 HOUR) AS eligible_at
FROM m10_retained_events
WHERE event_id = 1001
""", title="Oldest event and its next expiry boundary")
lab.dynamic_partition_status(
    "m10_retained_events", title="Scheduler after initial creation"
)
window_before_expiry = lab.partition_window_summary(
    "m10_retained_events", title="Hourly Partitions before the clock advances"
)


**Expected result:** event `1001` is present and `eligible_at` is the next whole-hour boundary in `Asia/Shanghai`. The five Partition ranges start three hours ago, two hours ago, one hour ago, this hour, and next hour. Three contain one event each; the previous-one-hour and future-hour Partitions are empty. The exact timestamps and names depend on when you start.

The event at `NOW() - INTERVAL 3 HOUR` is already three hours old, yet its complete hour remains online until the **next hourly boundary** and the subsequent scheduler check. Dynamic Partitioning removes a whole hour, not each row the instant it reaches age three hours.


## 3. Return after the hour and observe automatic expiry

**Pause here.** Keep the container and its named volumes; you may close the Notebook or restart its kernel. Return after the `eligible_at` time printed above, ideally allowing up to one FE scheduler check afterward. The default check interval is normally 600 seconds (ten minutes); the exact deletion time is not guaranteed. The Lab does **not** change the FE interval, alter `start`, or issue `DROP PARTITION` to accelerate this observation.

If you restarted the kernel, run only **Initialize the Lab** at the top, then continue here. Do not rerun Section 2 or you will reset the clock and the three events. The next cell reconnects and restores the session time zone. If you return before `eligible_at`, it shows when to come back and stops immediately instead of silently blocking for an hour. Once the boundary has passed, a live timer waits at most 15 minutes for the old Partition to disappear. Rerunning this wait after a timeout is safe; it does not modify the table.


In [ ]:
lab.start_container("doris")
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")
lab.execute("SET time_zone = 'Asia/Shanghai'")
lab.wait_for_hourly_partition_expiry(
    "m10_retained_events", event_id=1001,
    start_offset=-3, timeout_seconds=900,
)


**Expected result:** when you return after the displayed boundary, the timer shows elapsed `mm:ss` (up to 15 minutes) while checking the originally oldest Partition. It ends when that Partition disappears. If it was already removed before you came back, the helper says so without waiting. A run before the boundary stops with a specific return time. The timer measures waiting for the scheduler; it does **not** measure a three-hour row TTL.

Now run the **same** query for event `1001` and inspect the current Partition ranges. Neither setup nor this observation executed a manual Partition drop.


In [ ]:
after_expiry = lab.sql("""
SELECT event_time, event_id, note,
       DATE_ADD(DATE_TRUNC(event_time, 'hour'), INTERVAL 4 HOUR) AS eligible_at
FROM m10_retained_events
WHERE event_id = 1001
""", title="After hourly rollover: oldest event no longer online")
assert after_expiry.empty

lab.dynamic_partition_status(
    "m10_retained_events", title="Scheduler after automatic removal"
)
window_after_expiry = lab.partition_window_summary(
    "m10_retained_events", title="Hourly Partitions after automatic removal"
)


**Expected result:** the repeated `SELECT` returns **0 rows** for event `1001`; the Partition window no longer includes the hour shown for that event in Section 2. If you returned within the following hour, the newer event `1002` and current-hour event `1003` should still be online; returning much later can cause more whole hours to expire. The status card reports whether the scheduler is healthy, while the changed SQL answer and Partition ranges show what is actually online.

The disappearance proves a logical retention boundary, not immediate reclamation of every Backend (BE) storage byte. Recycle-bin retention and background cleanup are separate.

### Try an event outside the current window

An insert timestamp from four hours before the **current** hourly boundary falls outside `start = -3`. The next cell requires a focused missing-Partition error; it does not recreate the old Partition.


In [ ]:
late_write_error = lab.expect_statement_error("""
INSERT INTO m10_retained_events VALUES
    (DATE_SUB(DATE_TRUNC(NOW(), 'hour'), INTERVAL 4 HOUR),
     1099, 'outside hourly window')
""", contains="no partition for this tuple",
     title="Late event rejected outside the hourly window")

retained_rows = lab.sql("""
SELECT event_time, event_id, note
FROM m10_retained_events
ORDER BY event_time, event_id
""", title="Rows that remain online")
assert 1001 not in retained_rows["event_id"].astype(int).tolist()


**Expected result:** Doris reports `no partition for this tuple` for the four-hours-ago event. The online table does not contain event `1001` or the rejected event `1099`. Which newer events remain depends on how many whole hours passed while you were away.

A late event can still be accepted if its timestamp belongs to a retained Partition. A timestamp outside the window needs a separate business decision: reject it, route it elsewhere, or explicitly manage a historical Partition. Dynamic Partitioning does not make that decision automatically.


## 4. Recover a manually dropped Partition without a pre-recorded ID

Automatic retention and accidental deletion have different operating procedures. Use a separate manually partitioned table so that a scheduler cannot change the recovery example.

`p_recover` contains two known rows; `p_keep` contains one. **Without `FORCE`**, `ALTER TABLE ... DROP PARTITION p_recover` removes the Partition from the online table and puts it in the catalog recycle bin for a limited time, so `RECOVER PARTITION` can restore it. **With `FORCE`**, Doris bypasses this recovery path: that drop cannot be undone with `RECOVER`. This experiment deliberately omits `FORCE` and discovers the Partition ID only after deletion.

The setup cell uses `DROP TABLE IF EXISTS m10_recovery_events FORCE` only to reset an older copy of this Lab-owned table. That reset is not recoverable through the recycle bin and is **separate** from the recoverable Partition drop below.

By default, Doris keeps recoverable catalog entries for **86,400 seconds (24 hours)** after deletion; the Frontend (FE) setting `catalog_trash_expire_second` controls this period. The tested local Doris 4.1.3 sandbox currently also reports `86400`. Treat this as a limited opportunity to recover, not a guarantee: an administrator can change the setting, the entry can be purged, and a `FORCE` drop never enters this recovery path. Check `SHOW CATALOG RECYCLE BIN` before trying `RECOVER`.


In [ ]:
lab.execute("DROP TABLE IF EXISTS m10_recovery_events FORCE")
lab.execute("""
CREATE TABLE m10_recovery_events (
    event_date DATE NOT NULL,
    event_id BIGINT NOT NULL,
    revenue DECIMAL(12,2) NOT NULL,
    note VARCHAR(64) NOT NULL
)
DUPLICATE KEY(event_date, event_id)
PARTITION BY RANGE(event_date) (
    PARTITION p_recover VALUES [('2020-03-01'), ('2020-03-02')),
    PARTITION p_keep    VALUES [('2020-03-02'), ('2020-03-03'))
)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.insert("""
INSERT INTO m10_recovery_events VALUES
    ('2020-03-01', 2001, 19.95, 'recover me A'),
    ('2020-03-01', 2002, 29.95, 'recover me B'),
    ('2020-03-02', 2003,  9.95, 'keep me')
""", title="Insert three known recovery rows", expected_rows=3)

before_drop = lab.sql("""
SELECT event_date, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m10_recovery_events
GROUP BY event_date
ORDER BY event_date
""", title="Online answer before the accidental drop")
assert before_drop["event_count"].astype(int).tolist() == [2, 1];

**Expected result:** March 1 contains two rows totaling **49.90**; March 2 contains one row totaling **9.95**.

Next, drop `p_recover` **without `FORCE`** and query the recycle bin by name. Its metadata includes the dropped object type, table and Partition IDs, and deletion time. Adding `FORCE` here would make the next `RECOVER PARTITION` impossible for this drop. An earlier run may have left another entry named `p_recover`; look at the entries before choosing how to recover.


In [ ]:
lab.execute("""
ALTER TABLE m10_recovery_events
DROP PARTITION p_recover
""")

after_drop = lab.sql("""
SELECT event_date, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m10_recovery_events
GROUP BY event_date
ORDER BY event_date
""", title="Online answer after dropping p_recover")
assert after_drop["event_count"].astype(int).tolist() == [1]
assert str(after_drop.iloc[0]["total_revenue"]) == "9.95"

recycle_entries = lab.sql(
    "SHOW CATALOG RECYCLE BIN WHERE NAME = 'p_recover'",
    title="Recoverable entries named p_recover",
    columns=("Type", "Name", "TableId", "PartitionId", "DropTime"),
)
assert (recycle_entries["Type"].str.lower() == "partition").any()


**Expected result:** only the March 2 row remains queryable. The recycle-bin result includes a `Partition` named `p_recover` and its deletion time. It may include older entries with the same name if the Lab was run before. These IDs are displayed for diagnosis; the next command deliberately does not pass one.

Doris interprets a name-only `RECOVER PARTITION` as the most recently deleted metadata of that name. Here, the drop just happened in this isolated Lab, so that choice is intentional. In a real incident, check the `TableId`, `PartitionId`, and `DropTime` and specify the correct ID when the newest entry might not be the one you need. Recovery requires the entry to remain eligible in the recycle bin.


In [ ]:
lab.execute("""
RECOVER PARTITION p_recover
FROM m10_recovery_events
""")

recovered = lab.sql("""
SELECT event_date, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m10_recovery_events
GROUP BY event_date
ORDER BY event_date
""", title="Online answer after RECOVER PARTITION")
assert recovered["event_count"].astype(int).tolist() == [2, 1]
assert [str(value) for value in recovered["total_revenue"]] == ["49.90", "9.95"]
card("The just-dropped Partition and its two known rows are online again.",
     "ok", "Partition recovery verified")


**Expected result:** the two-date answer returns: March 1 with two rows and **49.90**, and March 2 with one row and **9.95**. This checks that the name-only recovery restored the intended logical data in this run.

The boundary matters:

- a manual Partition drop **without `FORCE`** enters the catalog recycle bin and can be recovered while its entry remains eligible;
- omitting the ID picks the last-deleted object of that name, not necessarily the object you intended in a real incident;
- `DROP PARTITION ... FORCE` bypasses the recoverable recycle bin, so `RECOVER PARTITION` cannot undo that forced drop;
- row-level `DELETE`, automatic retention, and an entire lost cluster must not be assumed to have the same recovery behavior;
- the recycle bin is not an independent backup. Cross-cluster disaster recovery requires a tested backup or replay from an authoritative source.


### Stop the Doris sandbox

Run this optional cell when you have finished and want to release the container's CPU and memory. It stops Doris but keeps the container, image, FE metadata volume, BE storage volume, and all `m10_*` objects. Skip it if you want to continue using the sandbox.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris
docker inspect --format 'container={{.State.Status}}' doris
""", title="Stop the Doris sandbox");

**Expected result:** Docker reports `container=exited`. The tables remain in the named volumes.

### Restart the Doris sandbox

This cell is idempotent: it can run when Docker Desktop and the container are stopped, starting, or already running. It starts or reuses the existing container, waits for health, reconnects, and checks the persisted compression and recovery tables.

In [ ]:
lab.start_container("doris")
lab.connect(container="doris", host="127.0.0.1", port=9030)
lab.execute("USE doris_course")

persisted = lab.sql("""
SELECT 'm10_events_none' AS object_name, COUNT(*) AS logical_rows
FROM m10_events_none
UNION ALL
SELECT 'm10_events_lz4f', COUNT(*) FROM m10_events_lz4f
UNION ALL
SELECT 'm10_events_zstd', COUNT(*) FROM m10_events_zstd
UNION ALL
SELECT 'm10_recovery_events', COUNT(*) FROM m10_recovery_events
ORDER BY object_name
""", title="Persisted Module 10 objects")
assert persisted.set_index("object_name")["logical_rows"].astype(int).to_dict() == {
    "m10_events_lz4f": 75_259,
    "m10_events_none": 75_259,
    "m10_events_zstd": 75_259,
    "m10_recovery_events": 3,
};

## Lab complete

You compared how NO_COMPRESSION, LZ4F, and ZSTD store the same logical data, related compression results to repeated values and shared byte prefixes, observed an hourly Dynamic Partition window remove a complete expired Partition, checked how a late event falls outside that window, and recovered an accidentally dropped Partition from the catalog recycle bin. These results describe the controlled Lab objects; they do not replace production workload testing or an independent backup. The stop and restart cells above preserve the `m10_*` tables for later inspection.

Official references: [Data compression](https://doris.apache.org/docs/4.x/table-design/column-compression/) · [SHOW DATA](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/data-and-status-management/SHOW-DATA/) · [Dynamic Partitioning](https://doris.apache.org/docs/4.x/table-design/data-partitioning/dynamic-partitioning/) · [SHOW CATALOG RECYCLE BIN](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/recycle/SHOW-CATALOG-RECYCLE-BIN/) · [RECOVER](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/recycle/RECOVER/)